In [1]:
#%pip install -U pandas

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
DATASET_CLEAN = Path("../data/processed/dataset_clean.csv")
MACRO_SUMMARY = Path("../data/processed/output/macro_summary.csv")
DATASET_WITH_MACROS = Path("../data/processed/output/dataset_with_clusters_and_macros.csv")

assert DATASET_CLEAN.exists(), f"Missing: {DATASET_CLEAN}"
assert MACRO_SUMMARY.exists(), f"Missing: {MACRO_SUMMARY}"
assert DATASET_WITH_MACROS.exists(), f"Missing: {DATASET_WITH_MACROS}"


In [4]:
df_clean = pd.read_csv(DATASET_CLEAN, encoding="utf-8", encoding_errors="replace")
macro_summary = pd.read_csv(MACRO_SUMMARY, encoding="utf-8", encoding_errors="replace")
df_all = pd.read_csv(DATASET_WITH_MACROS, encoding="utf-8", encoding_errors="replace")

print("df_clean shape:", df_clean.shape)
print("macro_summary shape:", macro_summary.shape)
print("df_all shape:", df_all.shape)

display(macro_summary.head())


df_clean shape: (3531, 17)
macro_summary shape: (10, 5)
df_all shape: (3531, 21)


,MacroId,MacroName,Count,MacroCategory,MacroSlug
0,2,Macro 2: medical imaging • tomography • analys...,945,Medical Imaging & Tomography,medical-imaging-tomography
1,5,Macro 5: chart • bar charts • scatterplots • p...,602,Charts & Standard Plots,charts-standard-plots
2,8,Macro 8: multidimensional scaling • scatterplo...,543,Multidimensional / Projection Views,multidimensional-projection
3,0,Macro 0: direct volume rendering • volume rend...,317,Volume Rendering & Transfer Functions,volume-rendering
4,3,Macro 3: flow field • flow fields • computatio...,313,Flow & Vector Field Visualization (CFD),flow-vector-fields-cfd


In [5]:
MACRO_WEB = {
    0: {"MacroCategory": "Volume Rendering & Transfer Functions", "MacroSlug": "volume-rendering"},
    1: {"MacroCategory": "Explainable AI & Deep Learning", "MacroSlug": "xai-deep-learning"},
    2: {"MacroCategory": "Medical Imaging & Tomography", "MacroSlug": "medical-imaging-tomography"},
    3: {"MacroCategory": "Flow & Vector Field Visualization (CFD)", "MacroSlug": "flow-vector-fields-cfd"},
    4: {"MacroCategory": "Molecular Visualization & Dynamics", "MacroSlug": "molecular-visualization-dynamics"},
    5: {"MacroCategory": "Charts & Standard Plots", "MacroSlug": "charts-standard-plots"},
    6: {"MacroCategory": "Visualization Design & Sketching", "MacroSlug": "viz-design-sketching"},
    7: {"MacroCategory": "Weather & Forecast Visualization", "MacroSlug": "weather-forecast-viz"},
    8: {"MacroCategory": "Multidimensional / Projection Views", "MacroSlug": "multidimensional-projection"},
    9: {"MacroCategory": "Sensemaking & Provenance", "MacroSlug": "sensemaking-provenance"},
}

missing = sorted(set(range(10)) - set(MACRO_WEB.keys()))
assert not missing, f"Missing macro ids in mapping: {missing}"

MACRO_WEB


{0: {'MacroCategory': 'Volume Rendering & Transfer Functions',
  'MacroSlug': 'volume-rendering'},
 1: {'MacroCategory': 'Explainable AI & Deep Learning',
  'MacroSlug': 'xai-deep-learning'},
 2: {'MacroCategory': 'Medical Imaging & Tomography',
  'MacroSlug': 'medical-imaging-tomography'},
 3: {'MacroCategory': 'Flow & Vector Field Visualization (CFD)',
  'MacroSlug': 'flow-vector-fields-cfd'},
 4: {'MacroCategory': 'Molecular Visualization & Dynamics',
  'MacroSlug': 'molecular-visualization-dynamics'},
 5: {'MacroCategory': 'Charts & Standard Plots',
  'MacroSlug': 'charts-standard-plots'},
 6: {'MacroCategory': 'Visualization Design & Sketching',
  'MacroSlug': 'viz-design-sketching'},
 7: {'MacroCategory': 'Weather & Forecast Visualization',
  'MacroSlug': 'weather-forecast-viz'},
 8: {'MacroCategory': 'Multidimensional / Projection Views',
  'MacroSlug': 'multidimensional-projection'},
 9: {'MacroCategory': 'Sensemaking & Provenance',
  'MacroSlug': 'sensemaking-provenance'}}

In [6]:
# Ensure MacroId is int (handles floats like 2.0)
macro_summary["MacroId"] = macro_summary["MacroId"].astype(int)

macro_summary["MacroCategory"] = macro_summary["MacroId"].map(lambda m: MACRO_WEB.get(m, {}).get("MacroCategory"))
macro_summary["MacroSlug"] = macro_summary["MacroId"].map(lambda m: MACRO_WEB.get(m, {}).get("MacroSlug"))

# Check if anything failed to map
unmapped = macro_summary[macro_summary["MacroCategory"].isna()]["MacroId"].unique().tolist()
if unmapped:
    print("WARNING: Unmapped MacroId(s) in macro_summary:", unmapped)

# Save enriched macro_summary back to the original filename
macro_summary.to_csv(MACRO_SUMMARY, index=False)
print("Updated file written:", MACRO_SUMMARY.resolve())

display(macro_summary)


Updated file written: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed/output/macro_summary.csv


,MacroId,MacroName,Count,MacroCategory,MacroSlug
0,2,Macro 2: medical imaging • tomography • analys...,945,Medical Imaging & Tomography,medical-imaging-tomography
1,5,Macro 5: chart • bar charts • scatterplots • p...,602,Charts & Standard Plots,charts-standard-plots
2,8,Macro 8: multidimensional scaling • scatterplo...,543,Multidimensional / Projection Views,multidimensional-projection
3,0,Macro 0: direct volume rendering • volume rend...,317,Volume Rendering & Transfer Functions,volume-rendering
4,3,Macro 3: flow field • flow fields • computatio...,313,Flow & Vector Field Visualization (CFD),flow-vector-fields-cfd
5,4,Macro 4: volume rendering • molecular modeling...,245,Molecular Visualization & Dynamics,molecular-visualization-dynamics
6,6,Macro 6: volume rendering • sketching • design...,165,Visualization Design & Sketching,viz-design-sketching
7,1,Macro 1: explainable machine learning • deep n...,153,Explainable AI & Deep Learning,xai-deep-learning
8,7,Macro 7: weather forecasting • weather predict...,126,Weather & Forecast Visualization,weather-forecast-viz
9,9,Macro 9: sensemaking process • sensemaking • p...,122,Sensemaking & Provenance,sensemaking-provenance


In [7]:

MERGE_KEY = "DOI"

if MERGE_KEY is None:
    raise ValueError(
        "No merge key detected. Set MERGE_KEY manually to a column that exists in BOTH files "
        "(dataset_clean.csv and dataset_with_clusters_and_macros.csv)."
    )

# Keep only what we need from df_all for the join
need_cols = [MERGE_KEY, "MacroId"]
missing_cols = [c for c in need_cols if c not in df_all.columns]
if missing_cols:
    raise ValueError(f"dataset_with_clusters_and_macros.csv is missing columns: {missing_cols}")

df_map = df_all[need_cols].copy()
df_map["MacroId"] = df_map["MacroId"].astype(int)

# Merge
before = len(df_clean)
df_out = df_clean.merge(df_map, on=MERGE_KEY, how="left", validate="m:1")
after = len(df_out)
assert before == after, "Row count changed after merge — check MERGE_KEY uniqueness."

# Create MacroCategory and drop MacroId (you asked: only the name in dataset_clean)
df_out["MacroCategory"] = df_out["MacroId"].map(
    lambda m: MACRO_WEB.get(int(m), {}).get("MacroCategory") if pd.notna(m) else np.nan
)
df_out = df_out.drop(columns=["MacroId"])

# Report missing macro category
missing_count = int(df_out["MacroCategory"].isna().sum())
print("Rows with missing MacroCategory:", missing_count, "/", len(df_out))
if missing_count > 0:
    print("Tip: check if some rows have no match in dataset_with_clusters_and_macros.csv on MERGE_KEY.")

df_out.to_csv(DATASET_CLEAN, index=False)

display(df_out.head())


Rows with missing MacroCategory: 0 / 3531


,Conference,Year,Title,DOI,PaperType,Abstract,AuthorNames-Deduped,AuthorAffiliation,InternalReferences,AuthorKeywords,AminerCitationCount,CitationCount_CrossRef,PubsCited_CrossRef,Downloads_Xplore,Award,GraphicsReplicabilityStamp,MacroCategory
0,Vis,2024,Interactive Design-of-Experiments: Optimizing ...,10.1109/tvcg.2024.3456356,J,The optimization of cooling systems is importa...,Rainer Splechtna;Majid Behravan;Mario Jelovic;...,"VRVis Research Center in Vienna, Austria;Virgi...",10.1109/tvcg.2013.124;10.1109/tvcg.2008.145;10...,Parameter space exploration,NaN,2.0,29.0,234.0,NaN,NaN,Multidimensional / Projection Views
1,Vis,2024,Towards Dataset-Scale and Feature-Oriented Eva...,10.1109/tvcg.2024.3456398,J,Recent advancements in Large Language Models (...,Sam Yu-Te Lee;Aryaman Bahukhandi;Dongyu Liu;Kw...,"University of California, USA;University of Ca...",10.1109/tvcg.2017.2743858;10.1109/tvcg.2017.27...,"Visual analytics,prompt engineering,,,text sum...",NaN,1.0,65.0,386.0,NaN,NaN,Charts & Standard Plots
2,Vis,2024,KNowNEt:Guided Health Information Seeking from...,10.1109/tvcg.2024.3456364,J,The increasing reliance on Large Language Mode...,Youfu Yan;Yu Hou;Yongkang Xiao;Rui Zhang;Qianw...,Department of Computer Science and Engineering...,10.1109/tvcg.2022.3209408;10.1109/tvcg.2023.33...,"Human-AI interactions,knowledge graph,,,conver...",NaN,1.0,60.0,632.0,HM,NaN,Charts & Standard Plots
3,Vis,2024,VisEval: A Benchmark for Data Visualization in...,10.1109/tvcg.2024.3456320,J,Translating natural language to visualization ...,Nan Chen;Yuge Zhang;Jiahang Xu;Kan Ren;Yuqing ...,"Microsoft Research, USA;Microsoft Research, US...",10.1109/infvis.2005.1532136;10.1109/tvcg.2015....,"Visualization evaluation,automatic visualizati...",NaN,1.0,75.0,625.0,BP,NaN,Charts & Standard Plots
4,Vis,2024,PUREsuggest: Citation-Based Literature Search ...,10.1109/tvcg.2024.3456199,J,Citations allow quickly identifying related re...,Fabian Beck 0001,"University of Bamberg, Germany",10.1109/tvcg.2015.2467757;10.1109/tvcg.2016.25...,"Scientific literature search,citation network ...",NaN,1.0,62.0,165.0,NaN,NaN,Medical Imaging & Tomography
